In [12]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
from datetime import date
from mlrl.boosting import Boomer

sys.path.append('..')
from data import Data
from options import Options
from boomerer import Boomerer

import pandas as pd
import builtins

In [ ]:
# loading the mnist datasets from the CSV formats downloaded from Kaggle
mnist_train_data = Data(filename='../bench_mnist_csv/mnist_train.csv', separator=',')
mnist_datanames = mnist_train_data.names[:-1]
mnist_train_data_np = np.asarray(mnist_train_data.samps, dtype=np.int32)
x_train_csv = mnist_train_data_np[:, 0:len(mnist_datanames)]
y_train_csv = mnist_train_data_np[:, len(mnist_datanames)]

# plt.imshow(x_train_csv[0].reshape([28, 28]), cmap='gray_r')
# print(y_train_csv[0])

In [13]:

def train_specific_boomer(dataset, max_rule, out_path, max_cond, wdigit=None):
    # preparing the command line for options
    # '-t' indicates training, '-n' indicates the n_estimator
    # 251118: add the option of limiting the weight digits
    if not wdigit is None:
        command_line = 'xxx -o ' + out_path + ' --train -n ' + str(max_rule) + ' -d ' + str(max_cond) + ' --wdigit ' + str(wdigit) + ' --drule ' + dataset
    else:
        command_line = 'xxx -o ' + out_path + ' --train -n ' + str(max_rule) + ' -d ' + str(max_cond) + ' --drule ' + dataset
    options = Options(command_line.split())

    if options.files:
        boomer = None
        if options.train:
            data = Data(filename=options.files[0], mapfile=options.mapfile,
                    separator=options.separator,
                    use_categorical=options.use_categorical)

            boomer = Boomerer(options, from_data=data)
        
            train_acc, test_acc, mod_fpath, num_rules = boomer.train()
            return train_acc, test_acc, mod_fpath, boomer.nb_features, boomer.num_class, boomer.num_instance, num_rules


In [ ]:
# 251011: prepare the name of the output folder
boomer_output = 'mnist_boomers_' + date.today().isoformat()[2:].replace('-', '')

# The default split ration is 20% for testing accuracy
nb_rules_candidate = [10, 50, 100, 200, 500, 800, 1000, 1500, 2000, 3000]
result_json = []
max_conds = [4, 5, 6]
dataset = '../bench_mnist_csv/mnist_train.csv'

# 251119: add the options of limiting the weight digits
wdigits = [None, 1,2, 3, 4, 5]

for nb_rule in nb_rules_candidate:
    for max_cond in max_conds:
        for wd in wdigits:
            # print("dataset {}, nb_rule {}, max_con {}".format(dataset, nb_rule, max_cond))
            train_acc, test_acc, mod_fpath, nb_feat, nb_c, nb_i, nb_rules = train_specific_boomer(dataset, nb_rule, boomer_output, max_cond, wd)
            dataset_n = dataset.split('/')[-1].split('.')[0]
            res_dict = {'dataset': dataset_n, 'n_estimator': nb_rule, 'max_depth': max_cond,\
                    'train_acc': train_acc, 'test_acc': test_acc, 'num_feat': nb_feat,\
                    'num_class': nb_c, 'num_instance': nb_i, 'num_identical_path': nb_rules, \
                    'wdigit': wd}
            result_json.append(res_dict)


In [ ]:

##### 260420!! Try small REs with high in max_conds
boomer_output = 'mnist_boomers_' + date.today().isoformat()[2:].replace('-', '')

# The default split ration is 20% for testing accuracy
nb_rules_candidate = [100, 200, 300, 400, 500, 600, 700, 800]
result_json = []
max_conds = [10, 30, 50]
dataset = '../bench_mnist_csv/mnist_train.csv'

wdigits = [None, 1, 2]

for nb_rule in nb_rules_candidate:
    for max_cond in max_conds:
        for wd in wdigits:
            # print("dataset {}, nb_rule {}, max_con {}".format(dataset, nb_rule, max_cond))
            train_acc, test_acc, mod_fpath, nb_feat, nb_c, nb_i, nb_rules = train_specific_boomer(dataset, nb_rule, boomer_output, max_cond, wd)
            dataset_n = dataset.split('/')[-1].split('.')[0]
            res_dict = {'dataset': dataset_n, 'n_estimator': nb_rule, 'max_depth': max_cond,\
                    'train_acc': train_acc, 'test_acc': test_acc, 'num_feat': nb_feat,\
                    'num_class': nb_c, 'num_instance': nb_i, 'num_identical_path': nb_rules, \
                    'wdigit': wd}
            result_json.append(res_dict)


In [28]:
import json
json_file_name = 'result' + boomer_output[5:] + '.json'
if os.path.isfile(json_file_name):
    os.remove(json_file_name)
with open(json_file_name, 'w') as f:
    json.dump(result_json, f, indent=2)

### Generating Tables of Results concerning prediction quality !!

In [20]:
def get_df_infos(df, max_depth, n_estimator, dataset, wdigit=None):
    assert (max_depth in set(df['max_depth'])) and (n_estimator in set(df['n_estimator']) and dataset in set(df['dataset']))
    if not wdigit is None:
        assert (wdigit in set(df['wdigit']))
        df_infos = df[(df['max_depth'] == max_depth) & (df['n_estimator'] == n_estimator)\
                    & (df['dataset'] == dataset) & (df['wdigit'] == wdigit)]
    else:
        df_infos = df[(df['max_depth'] == max_depth) & (df['n_estimator'] == n_estimator) & (df['dataset'] == dataset)]
    assert df_infos.shape[0] == 1
    return df_infos

def value2str(v):
    if isinstance(v, np.float64):
        return f'{v:.3f}'
    elif isinstance(v, np.int64):
        return str(v)
    else:
        return v

def prepare_table_by_bench(df, out_path):
    cols_selected = [ 'num_identical_path', 'train_acc', 'test_acc']
    d_info = 'mnist_train'

    # 1st col (m_depth) => 2nd col (num_estimator) => Boomer (train_acc/test_acc/identical_path)
    num_estimators = sorted(set(df['n_estimator']))
    with builtins.open(out_path, 'w') as f:
        for m_depth in sorted(set(df['max_depth'])):
            # prepare each multirow cell
            f.write('\\multirow{' + str(len(num_estimators)) + '}{*}{' + str(m_depth) + '}\n')
            for n in num_estimators:
                f.write('& ' + str(n) + ' & ')
                df_infos = get_df_infos(df, m_depth, n, d_info)
                str_row = ''
                for col in cols_selected:
                    str_row += value2str(df_infos.iloc[-1][col]) + ' & '
                f.write(str_row[:-2] + '\\\\\n')
            f.write('\\hline \n')

def prepare_table_by_bench_with_wd(df, out_path):
    cols_selected = ['train_acc', 'test_acc']
    d_info = 'mnist_train'

    # 1st col (m_depth) => 2nd col (num_estimator) => Boomer (train_acc/test_acc) for each w_digit
    num_estimators = sorted(set(df['n_estimator']))
    wdigits = sorted(set(df['wdigit']))
    wdigits = wdigits[:3]
    print(wdigits)
    
    with builtins.open(out_path, 'w') as f:
        for m_depth in sorted(set(df['max_depth'])):
            # prepare each multirow cell
            f.write('\\multirow{' + str(len(num_estimators)) + '}{*}{' + str(m_depth) + '}\n')
            for n in num_estimators:
                f.write('& ' + str(n) + ' & ')
                df_orig_infos = get_df_infos(df, m_depth, n, d_info, wdigit=-1)
                
                # 260119: add the num_identical path info
                f.write(str(df_orig_infos.iloc[-1]['num_identical_path']) + ' & ')

                str_row = ''
                for wd in wdigits:
                    df_infos = get_df_infos(df, m_depth, n, d_info, wdigit=wd)
                    for col in cols_selected:
                        # presenting the differenc to the orig instead of value
                            if wd != -1.0:
                                str_row += value2str(df_infos.iloc[-1][col] - df_orig_infos.iloc[-1][col]) + ' & '
                            else:
                                str_row += value2str(df_infos.iloc[-1][col]) + ' & '

                f.write(str_row[:-2] + '\\\\\n')
            f.write('\\hline \n')


In [ ]:
# 251103: generating the table of Boomer models for MNIST dataset
json_file_name = './result_boomers_251119.json' # containing original and different wds (orig, 1 to 5)
df_mnist_boomers = pd.read_json(json_file_name)
df_mnist_boomers = df_mnist_boomers.where(pd.notnull(df_mnist_boomers), -1)

#############################
# prepare_table_by_bench(df_mnist_boomers, json_file_name[:-4] + '.tex')
prepare_table_by_bench_with_wd(df_mnist_boomers, json_file_name[:-5] + '.tex')

In [29]:
# 260504: the rable of Boomer model for MNIST dataset with new results
# json_file_name = './result_boomers_260430.json' 
json_file_name = './result_boomers_260505.json' 
df_mnist_boomers = pd.read_json(json_file_name)
df_mnist_boomers = df_mnist_boomers.where(pd.notnull(df_mnist_boomers), -1)

prepare_table_by_bench_with_wd(df_mnist_boomers, json_file_name[:-5] + '.tex')

[-1.0, 1.0, 2.0]


AssertionError: 

In [ ]:
#############################
# For XGBoost
# 260115: add the tables for XGBoost in MNIST
json_file_name_x = './result_xgboost_260116.json'
df_mnist_xgboost = pd.read_json(json_file_name_x)
df_mnist_xgboost = df_mnist_xgboost.where(pd.notnull(df_mnist_xgboost), -1)
prepare_table_by_bench(df_mnist_xgboost, json_file_name_x[:-5] + '.tex')

### Generating Instances to Explain!! (MaxSAT encoding Version)

In [ ]:
# 251103: add the codes to explain different digit instances
import random
import itertools

# preparing 10 random digit instances from 0 to 9 => random chosen from the dataset
mnist_image_x_str = []
digits_class = list(set(y_train_csv))

for c in digits_class:
    indices_c = [i for i, x in enumerate(y_train_csv) if x == c]
    assert len(indices_c) > 0
    idc = random.Random(2025).choice(indices_c)
    mnist_image_x_str.append(','.join(map(str, x_train_csv[idc])))

# preparing the models to be explained by the digits instances
nb_rules_explained = nb_rules_candidate[5:]
max_conds_explained = max_conds[1:]

mod_base_path = './mnist_boomers_251013/mnist_train'
mod_paths = []
for nb, mcond in itertools.product(nb_rules_explained, max_conds_explained):
    mod_paths.append(os.path.join(mod_base_path, 'mnist_train_maxrule_' + str(nb) + '_maxcond_' + \
                    str(mcond) + '_testsplit_0.2.mod.pkl'))

# preparing the list of commands
commands = []
for mnist_i, mod in itertools.product(mnist_image_x_str, mod_paths):
    command_str = 'python3 ../Exp2_Explanation_BoostRules/xreasonBR.py -s g3 -z -X abd -R lin -u -N 1 -vv -e mx -x \''+\
                    mnist_i + '\' ' + mod + ' > Axp_mnist_processed/m' + str(mnist_image_x_str.index(mnist_i)) + '_' 
    mod_info = mod.split('/')[-1][:-22]
    command_str += mod_info[12:] + '.txt &'
    commands.append(command_str)

if os.path.exists('commands.txt'):
    os.remove('commands.txt')
with open('commands.txt', 'w') as f:
    for command in commands:
        f.write(command + '\n')


In [ ]:
def imshow_digits_with_expl(mnist_image, out_texts, t='Axp'):
    plt.figure()
    for o_text in out_texts:
        hypos_left, rtime = 0, 0.0
        expl_x, expl_y = [], []
        with open(o_text, 'r') as f:
            for line in f.readlines():
                if line.startswith('| Explained:'):
                    expl_str_list = line.split('"')[1].strip(' ').split(' AND ')
                    expl_str_list[0] = expl_str_list[0].split('IF ')[-1]
                    expl_str_list[-1] = expl_str_list[-1].split(' THEN ')[0]
                    for expl_str in expl_str_list:
                        if t == 'Axp':
                            k, v = expl_str.split(' == ')
                        if t == 'Cxp':
                            k, v = expl_str.split(' != ')
                        k_x, k_y = k.split('x')
                        expl_y.append(int(k_x) - 1)
                        expl_x.append(int(k_y) - 1)
                elif line.strip(' ').startswith('# hypos'):
                    hypos_left = int(line.split(':')[-1].strip(' '))
                    assert len(expl_x) == hypos_left
                elif line.strip(' ').startswith('rtime'):
                    rtime = float(line.split(':')[-1].strip(' '))
                else:
                    continue
        print(t + ': ' + o_text + ', #hypos: ' + str(hypos_left) + ', rtime: ' + str(rtime))
        # printing the image and points of Axps extracted
        plt.gray()
        plt.imshow(mnist_image.reshape([28, 28]), cmap='gray')
        if t == 'Axp':
            plt.scatter(expl_x, expl_y, color='red', s=10, marker='x')
        if t == 'Cxp':
            plt.scatter(expl_x, expl_y, color='blue', s=10, marker='o')
        # saving the figure with infos
        fig_path = o_text.split('/')[-1][:-4] + '-' + str(hypos_left) + '-' + str(rtime) + '.png'
        plt.savefig(os.path.join('./mnist_image_axp_processed', fig_path))
        plt.show()
        plt.close()


In [ ]:
# imshowing the digits and the Axp obtained
axp_mnist_bpath = './Axp_mnist_processed'
axp_out_txt = sorted([os.path.join(axp_mnist_bpath, p) for p in os.listdir(axp_mnist_bpath)])

for out_id, mnist_str in enumerate(mnist_image_x_str):
    mnist_int = np.asarray([int(i) for i in mnist_str.split(',')])

    axp_out_id = [a for a in axp_out_txt if int(a.split('/')[-1][1]) == out_id]
    imshow_digits_with_expl(mnist_int, axp_out_id, t='Axp')

In [ ]:
# explaining the first image of mnist
# mnist_image_x = np.asarray([0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,18,18,18,126,136,175,26,166,255,247,127,0,0,0,0,0,0,0,0,0,0,0,0,30,36,94,154,170,253,253,253,253,253,225,172,253,242,195,64,0,0,0,0,0,0,0,0,0,0,0,49,238,253,253,253,253,253,253,253,253,251,93,82,82,56,39,0,0,0,0,0,0,0,0,0,0,0,0,18,219,253,253,253,253,253,198,182,247,241,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,80,156,107,253,253,205,11,0,43,154,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,14,1,154,253,90,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,139,253,190,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,11,190,253,70,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,35,241,225,160,108,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,81,240,253,253,119,25,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,45,186,253,253,150,27,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,16,93,252,253,187,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,249,253,249,64,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,46,130,183,253,253,207,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,39,148,229,253,253,253,250,182,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,24,114,221,253,253,253,253,201,78,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,23,66,213,253,253,253,253,198,81,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,18,171,219,253,253,253,253,195,80,9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,55,172,226,253,253,253,253,244,133,11,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,136,253,253,253,212,135,132,16,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0])
mnist_image_x_0 = x_train_csv[0]

# outputs => to be updated!
axp_output_bpath = '../Exp2_Explanation_BoostRules/Axp_out_mnist'
axp_outputs_text = ['m0_out_mnist_10_6.txt']
                # 'm0_out_mnist_50_6.txt',
                # 'm0_out_mnist_200_6.txt', 
                # 'm0_out_mnist_500_4.txt', 'm0_out_mnist_500_5.txt', 
                # 'm0_out_mnist_800_4.txt']
axp_outputs = [os.path.join(axp_output_bpath, i) for i in axp_outputs_text]
# axp_outputs.append(os.path.join('../Exp2_Explanation_BoostRules/Axp_out_mnist_notsmallest', axp_outputs_text[0]))

cxp_output_bpath = '../Exp2_Explanation_BoostRules/Cxp_out_mnist'
cxp_outputs_text = ['m0_out_mnist_10_6.txt']
cxp_outputs = [os.path.join(cxp_output_bpath, i) for i in cxp_outputs_text]

# showing the Axps extracted in the image => x_0, image of 5
imshow_digits_with_expl(mnist_image_x_0, axp_outputs, t='Axp')
imshow_digits_with_expl(mnist_image_x_0, cxp_outputs, t='Cxp')

